In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])


0

In [2]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "new_models"
DATA_DIR    = PROJECT_ROOT / "data" / "raw"
print("Project root:", PROJECT_ROOT)

Project root: c:\Users\user\OneDrive\Desktop\Praktikum\Final-Project\new_equity_forecasting_project


In [3]:
ref = pd.read_csv(RESULTS_DIR / "all_predictions.csv")
ref["Date"] = pd.to_datetime(ref["Date"], errors="coerce")
ref["cutoff_date"] = pd.to_datetime(ref["cutoff_date"], errors="coerce")
cutoffs = sorted(ref["cutoff_date"].dropna().dt.normalize().unique())
tickers = sorted(ref["Ticker"].unique())
print(f"Cutoffs : {len(cutoffs)}")
print(f"Tickers : {len(tickers)}")
print(f"Range   : {cutoffs[0].date()} -> {cutoffs[-1].date()}")

Cutoffs : 18
Tickers : 160
Range   : 2024-11-20 -> 2026-04-03


In [4]:
price_data = {}
for ticker in tickers:
    path = DATA_DIR / f"{ticker}.csv"
    if not path.exists():
        continue
    df = pd.read_csv(path, parse_dates=["Date"], index_col="Date")
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    close_col = next((c for c in df.columns if "close" in c and "adj" not in c), None)
    if close_col is None:
        continue
    df = df.rename(columns={close_col: "close"})
    for col in ["open", "high", "low"]:
        match = next((c for c in df.columns if col in c), None)
        if match and match != col:
            df = df.rename(columns={match: col})
    df = df.sort_index().dropna(subset=["close"])
    price_data[ticker] = df

print(f"Loaded {len(price_data)} tickers")

Loaded 49 tickers


In [5]:
def make_features(df, date):
    hist = df[df.index <= pd.Timestamp(date)].copy()
    if len(hist) < 80:
        return None
    cl = hist["close"]

    ret_1m = float(cl.pct_change(21).iloc[-1])
    ret_3m = float(cl.pct_change(63).iloc[-1])
    ret_6m = float(cl.pct_change(126).iloc[-1]) if len(cl) >= 126 else ret_3m
    vol    = float(cl.pct_change().iloc[-21:].std() * np.sqrt(252))

    d    = cl.diff()
    gain = d.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = (-d).clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    rsi  = float((100 - 100 / (1 + gain / loss.replace(0, np.nan))).iloc[-1])

    cur   = float(cl.iloc[-1])
    ema20 = float(cl.ewm(span=20, adjust=False).mean().iloc[-1])
    ema50 = float(cl.ewm(span=50, adjust=False).mean().iloc[-1])
    price_vs_ema20 = (cur - ema20) / ema20
    ema_ratio      = ema20 / ema50 - 1

    if "high" in hist.columns and "low" in hist.columns:
        tr  = pd.concat([hist["high"] - hist["low"],
                         (hist["high"] - cl.shift()).abs(),
                         (hist["low"]  - cl.shift()).abs()], axis=1).max(axis=1)
        atr_norm = float(tr.ewm(alpha=1/14, adjust=False).mean().iloc[-1] / cur)
    else:
        atr_norm = vol / np.sqrt(252)

    feats = [ret_1m, ret_3m, ret_6m, vol, rsi, price_vs_ema20, ema_ratio, atr_norm]
    if any(not np.isfinite(v) for v in feats):
        return None
    return feats

In [6]:
def barrier_label(df, date, h_months, upper_pct, lower_pct):
    t     = pd.Timestamp(date)
    hist  = df[df.index <= t]["close"]
    if hist.empty:
        return np.nan
    start = float(hist.iloc[-1])
    fwd   = df[df.index > t]["close"].head(h_months * 22)
    if len(fwd) < 3:
        return np.nan
    for price in fwd:
        ret = price / start - 1
        if ret >=  upper_pct: return  1
        if ret <= -lower_pct: return -1
    final = float(fwd.iloc[-1]) / start - 1
    return 1 if final > 0 else (-1 if final < 0 else 0)

In [8]:
UPPER_H1 = 0.025
LOWER_H1 = 0.025
UPPER_H5 = 0.06
LOWER_H5 = 0.06

all_rows = []

for h_months, h_label, upper, lower in [
    (1, 1, UPPER_H1, LOWER_H1),
    (5, 5, UPPER_H5, LOWER_H5),
]:
    print(f"\n=== Horizon h={h_label} | barriers +/-{upper:.1%} ===")

    for i, cutoff in enumerate(cutoffs):
        train_end = cutoff - pd.DateOffset(months=2)
        X_train, y_train = [], []

        for ticker, df in price_data.items():
            hist = df[df.index <= train_end]
            if len(hist) < 100:
                continue
            sample_dates = hist.resample("MS").last().index[-30:]
            for sd in sample_dates:
                feats = make_features(df, sd)
                if feats is None:
                    continue
                lbl = barrier_label(df, sd, h_months, upper, lower)
                if np.isnan(lbl):
                    continue
                X_train.append(feats)
                y_train.append(int(lbl))

        if len(X_train) < 30:
            print(f"  [{i+1}/{len(cutoffs)}] {cutoff.date()} skipped ({len(X_train)} samples)")
            continue

        clf = RandomForestClassifier(n_estimators=150, max_depth=6,
                                     min_samples_leaf=5, random_state=42, n_jobs=-1)
        clf.fit(X_train, y_train)
        classes = list(clf.classes_)

        n_pred = 0
        for ticker, df in price_data.items():
            feats = make_features(df, cutoff)
            if feats is None:
                continue
            proba  = clf.predict_proba([feats])[0]
            p_up   = proba[classes.index( 1)] if  1 in classes else 0.0
            p_down = proba[classes.index(-1)] if -1 in classes else 0.0
            signal = float(p_up - p_down)

            ref_row = ref[
                (ref["Ticker"] == ticker) &
                (ref["cutoff_date"].dt.normalize() == cutoff) &
                (ref["Horizon"] == h_label)
            ]
            if ref_row.empty:
                continue

            all_rows.append({
                "Date":         ref_row["Date"].iloc[0],
                "Ticker":       ticker,
                "Model":        "BarrierRF",
                "Horizon":      h_label,
                "y_true":       float(ref_row["y_true"].iloc[0]),
                "y_pred":       signal,
                "cutoff_date":  cutoff.strftime("%Y-%m-%d"),
                "last_close":   np.nan,
                "pred_close":   np.nan,
                "actual_close": np.nan,
            })
            n_pred += 1

        print(f"  [{i+1}/{len(cutoffs)}] {cutoff.date()} — {len(X_train)} train samples, {n_pred} predictions")

print(f"\nTotal predictions: {len(all_rows)}")



=== Horizon h=1 | barriers +/-2.5% ===
  [1/18] 2024-11-20 — 1470 train samples, 49 predictions
  [2/18] 2024-12-19 — 1470 train samples, 49 predictions
  [3/18] 2025-01-17 — 1470 train samples, 49 predictions
  [4/18] 2025-02-17 — 1470 train samples, 49 predictions
  [5/18] 2025-03-18 — 1470 train samples, 49 predictions
  [6/18] 2025-04-16 — 1470 train samples, 49 predictions
  [7/18] 2025-05-15 — 1470 train samples, 49 predictions
  [8/18] 2025-06-13 — 1470 train samples, 49 predictions
  [9/18] 2025-07-14 — 1470 train samples, 49 predictions
  [10/18] 2025-08-12 — 1470 train samples, 49 predictions
  [11/18] 2025-09-10 — 1470 train samples, 49 predictions
  [12/18] 2025-10-09 — 1470 train samples, 49 predictions
  [13/18] 2025-11-07 — 1470 train samples, 49 predictions
  [14/18] 2025-12-08 — 1470 train samples, 49 predictions
  [15/18] 2026-01-06 — 1470 train samples, 49 predictions
  [16/18] 2026-02-04 — 1470 train samples, 49 predictions
  [17/18] 2026-03-05 — 1470 train samples

In [9]:
barrier_df = pd.DataFrame(all_rows)
out = RESULTS_DIR / "barrier_predictions.csv"
barrier_df.to_csv(out, index=False)
print(f"Saved  : {out}")
print(f"Shape  : {barrier_df.shape}")
print(barrier_df[["Model","Horizon","y_true","y_pred"]].describe().round(4))
     

Saved  : c:\Users\user\OneDrive\Desktop\Praktikum\Final-Project\new_equity_forecasting_project\results\new_models\barrier_predictions.csv
Shape  : (1764, 10)
         Horizon     y_true     y_pred
count  1764.0000  1764.0000  1764.0000
mean      3.0000     0.0042     0.0979
std       2.0006     0.0356     0.1564
min       1.0000    -0.2685    -0.4814
25%       1.0000    -0.0113     0.0012
50%       3.0000     0.0035     0.0873
75%       5.0000     0.0184     0.1793
max       5.0000     0.2789     0.7337


In [10]:
from scipy.stats import spearmanr
print("=== Quick RankIC check ===")
for h in [1, 5]:
    sub  = barrier_df[barrier_df["Horizon"] == h]
    rics = []
    for cutoff, g in sub.groupby("cutoff_date"):
        g = g.dropna(subset=["y_true", "y_pred"])
        if len(g) >= 5:
            r, _ = spearmanr(g["y_pred"], g["y_true"])
            if np.isfinite(r):
                rics.append(r)
    print(f"  h={h}: mean RankIC = {np.mean(rics):.4f}  ({len(rics)} cutoffs)")
print("\nRe-run notebook 04_final_analysis.ipynb to add BarrierRF to all charts.")
     

=== Quick RankIC check ===
  h=1: mean RankIC = 0.0386  (17 cutoffs)
  h=5: mean RankIC = 0.0278  (18 cutoffs)

Re-run notebook 04_final_analysis.ipynb to add BarrierRF to all charts.
